# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load and explore the [FAIR\^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library, leveraging Croissant schema definitions and unique `@id` references for all data-related entities.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load and print the dataset metadata using `mlcroissant`. All metadata access uses dataset attributes or methods.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Published: {meta.date_published if hasattr(meta, 'date_published') else 'N/A'}\n")
print(f"Authors IDs:")
if hasattr(meta, 'author'):
    for author in meta.author:
        if hasattr(author, '@id'):
            print(" -", author['@id'])

## 2. Data Overview
Explore the available record sets and their corresponding field and column `@id`s. This is crucial for referencing entities throughout the rest of the notebook.

In [ ]:
# List available RecordSets by @id for this dataset
print("Available record sets (by @id):\n")
if hasattr(meta, 'record_set'):
    record_sets = meta.record_set
else:
    record_sets = getattr(meta, 'recordSets', [])  # fallback

if not record_sets:
    print("- No RecordSets found on metadata. Attempting to load from schema...")
    # Try fetching via the Croissant interface
    record_sets = dataset.record_sets  # Supported in recent mlcroissant

record_set_ids = []
for rs in record_sets:
    rsid = rs['@id'] if isinstance(rs, dict) and '@id' in rs else (rs['id'] if isinstance(rs, dict) and 'id' in rs else str(rs))
    print(f"- {rsid}")
    record_set_ids.append(rsid)

    # List fields for each record set
    if isinstance(rs, dict) and 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields (by @id):")
        for f in fields:
            fid = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
            print(f"    - {fid}")

## 3. Data Extraction
Load all records from each available record set into DataFrames. These will be used for further exploration. All record set and field access is by `@id`.

We will preview the top columns and a snippet of the main record set.

In [ ]:
# If you know the main record set @id, you can specify it, otherwise we use the first
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
else:
    selected_record_set_id = None

print(f"\nExtracting data from RecordSet: {selected_record_set_id}\n")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet {record_set_id} columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print(f"No records found for RecordSet {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (e.g., protein molecular weight or abundance) to illustrate filtering, normalization, and grouping. Make sure to use the actual `@id` name from the DataFrame columns. Typical column names may include `MW`, `NormalizedAbundance`, `PeptideCount`, etc. We'll attempt to infer a relevant numeric field.

> **If you encounter a warning about chained assignment in pandas, this is OK for demonstration. In production, use `.copy()`.**

In [ ]:
import numpy as np
record_set_id = selected_record_set_id
df = dataframes.get(record_set_id, pd.DataFrame())

# Choose a numeric column by checking DataFrame dtypes
possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if not possible_numeric_fields:
    # Try to convert common numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    numeric_field_id = None

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'iufc' else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field if available
    possible_categoricals = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
    group_field_id = None
    if possible_categoricals:
        group_field_id = possible_categoricals[0]

    if group_field_id:
        print(f"\nGrouped mean for {numeric_field_id} by {group_field_id}:\n")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped.head())
else:
    print("No numeric fields were found for EDA in this RecordSet.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the relationship with a chosen categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/), loading tabular data, referencing entities by their Croissant schema `@id`, and performing basic exploratory data analysis.

- All navigation and access to data is done by unique `@id`s, ensuring reliable reference for all elements.
- Numeric and categorical fields were detected automatically; for deeper exploration, refer to the Croissant documentation or schema for field descriptions and types.
- This template can be adapted for any Croissant-conformant dataset for FAIR, reproducible data science.